# Initial Libraies

In [1]:
!pip install job-shop-lib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from job_shop_lib.visualization.gantt import plot_gantt_chart
from job_shop_lib import JobShopInstance, Operation
from job_shop_lib.dispatching import (
    ReadyOperationsFilterType,
    create_composite_operation_filter,
)
from job_shop_lib.dispatching.rules import (
    DispatchingRuleSolver,
    DispatchingRuleType,
)

from job_shop_lib.benchmarking import load_benchmark_instance

from collections import defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
import random
import math
from typing import Any

# Instance generation function

In [3]:
def validate_instance_dict(instance_dict: dict[str, Any]) -> None:
    required_keys = {"name", "duration_matrix", "machines_matrix"}
    missing = required_keys - set(instance_dict.keys())
    if missing:
        raise ValueError(f"Missing required keys in instance_dict: {missing}")

    duration_matrix = instance_dict["duration_matrix"]
    machines_matrix = instance_dict["machines_matrix"]

    if not duration_matrix or not machines_matrix:
        raise ValueError("duration_matrix and machines_matrix must be non-empty.")

    if len(duration_matrix) != len(machines_matrix):
        raise ValueError("duration_matrix and machines_matrix must have the same number of jobs.")

    num_ops = len(duration_matrix[0])
    for j, (dur_row, mac_row) in enumerate(zip(duration_matrix, machines_matrix)):
        if len(dur_row) != len(mac_row):
            raise ValueError(f"Row {j} has inconsistent lengths in duration_matrix and machines_matrix.")
        if len(dur_row) != num_ops:
            raise ValueError("All jobs must have the same number of operations.")


In [4]:
def get_num_jobs(instance_dict: dict[str, Any]) -> int:
    return len(instance_dict["duration_matrix"])


def get_num_machines(instance_dict: dict[str, Any]) -> int:
    return max(max(row) for row in instance_dict["machines_matrix"]) + 1


def estimate_horizon(instance_dict: dict[str, Any], horizon_factor: float = 1.0) -> int:
    """
    Conservative default horizon: factor * total processing load.
    """
    total_processing = sum(sum(row) for row in instance_dict["duration_matrix"])
    return max(1, int(math.ceil(horizon_factor * total_processing)))

In [5]:
def sample_interarrival(rng: random.Random, spec: dict[str, Any]) -> float:
    """
    Supported specs:
      {"dist": "exponential", "mtbf": 300}
      {"dist": "weibull", "shape": 1.5, "scale": 300}
      {"dist": "deterministic", "value": 300}
    """
    dist = spec.get("dist", "exponential")

    if dist == "exponential":
        mtbf = spec["mtbf"]
        if mtbf <= 0:
            raise ValueError("For exponential interarrival, mtbf must be > 0.")
        return rng.expovariate(1.0 / mtbf)

    elif dist == "weibull":
        shape = spec["shape"]
        scale = spec["scale"]
        if shape <= 0 or scale <= 0:
            raise ValueError("For weibull interarrival, shape and scale must be > 0.")
        return rng.weibullvariate(alpha=scale, beta=shape)

    elif dist == "deterministic":
        value = spec["value"]
        if value <= 0:
            raise ValueError("For deterministic interarrival, value must be > 0.")
        return float(value)

    else:
        raise ValueError(f"Unsupported interarrival distribution: {dist}")


def sample_duration(rng: random.Random, spec: dict[str, Any]) -> int:
    """
    Supported specs:
      {"dist": "fixed", "value": 40}
      {"dist": "deterministic", "value": 40}
      {"dist": "uniform", "low": 20, "high": 50}
      {"dist": "lognormal", "mean": 40, "sigma": 0.35}

    For lognormal:
      - mean is interpreted as the desired arithmetic mean approximately
      - sigma is the log-space std dev
    """
    dist = spec.get("dist", "fixed")

    if dist in {"fixed", "deterministic"}:
        value = spec["value"]
        if value <= 0:
            raise ValueError("For fixed/deterministic duration, value must be > 0.")
        return max(1, int(round(value)))

    elif dist == "uniform":
        low = spec["low"]
        high = spec["high"]
        if low <= 0 or high <= 0 or low > high:
            raise ValueError("For uniform duration, require 0 < low <= high.")
        return rng.randint(int(low), int(high))

    elif dist == "lognormal":
        mean = spec["mean"]
        sigma = spec["sigma"]
        if mean <= 0 or sigma <= 0:
            raise ValueError("For lognormal duration, mean and sigma must be > 0.")

        # Convert arithmetic mean + log-space sigma to mu
        mu = math.log(mean) - 0.5 * sigma * sigma
        x = rng.lognormvariate(mu, sigma)
        return max(1, int(round(x)))

    else:
        raise ValueError(f"Unsupported duration distribution: {dist}")


In [6]:
def resolve_machine_model(
    machine_id: int,
    machine_models: dict[Any, dict[str, Any]] | None = None,
) -> dict[str, Any]:
    """
    Resolution order:
      1. machine_models[machine_id]
      2. machine_models["default"]
      3. built-in fallback default
    """
    builtin_default = {
        "interarrival": {"dist": "exponential", "mtbf": 300},
        "duration": {"dist": "fixed", "value": 40},
        "initial_offset": 0.0,
    }

    if machine_models is None:
        return builtin_default

    if machine_id in machine_models:
        model = dict(builtin_default)
        model.update(machine_models.get("default", {}))
        model.update(machine_models[machine_id])
        return model

    if "default" in machine_models:
        model = dict(builtin_default)
        model.update(machine_models["default"])
        return model

    return builtin_default

In [7]:
def generate_holes_for_machine(
    machine_id: int,
    horizon: int,
    rng: random.Random,
    machine_models: dict[Any, dict[str, Any]] | None,
    without_fixed_dates: bool,
    allow_overlap: bool = False,
) -> list[tuple[int, int]]:
    """
    Generates a list of holes (start, duration) for one machine.

    Logic:
      - sample first hole start after an optional initial_offset
      - then repeatedly sample interarrival times
      - stop when next start >= horizon

    If without_fixed_dates=True:
      - starts are output as -1
      - number and durations are still sampled using the same process
    """
    model = resolve_machine_model(machine_id, machine_models)
    interarrival_spec = model["interarrival"]
    duration_spec = model["duration"]
    initial_offset = float(model.get("initial_offset", 0.0))

    holes: list[tuple[int, int]] = []

    t = initial_offset

    while True:
        delta = sample_interarrival(rng, interarrival_spec)
        t += delta
        start = int(round(t))

        if start >= horizon:
            break

        duration = sample_duration(rng, duration_spec)

        if allow_overlap or not holes:
            holes.append((start, duration))
        else:
            prev_start, prev_dur = holes[-1]
            prev_end = prev_start + prev_dur

            if start < prev_end:
                start = prev_end

            if start < horizon:
                holes.append((start, duration))
            else:
                break

    if without_fixed_dates:
        return [(-1, duration) for _, duration in holes]

    return holes


In [8]:
def generate_machine_holes(
    instance_dict: dict[str, Any],
    seed: int,
    without_fixed_dates: bool = False,
    horizon: int | None = None,
    horizon_factor: float = 1.0,
    machine_models: dict[Any, dict[str, Any]] | None = None,
    allow_overlap: bool = False,
) -> dict[int, list[tuple[int, int]]]:
    """
    Generate holes for all machines.

    Parameters
    ----------
    instance_dict
        Dictionary from benchmark_instance.to_dict().
    seed
        Random seed.
    without_fixed_dates
        If True, output holes as (-1, duration).
    horizon
        Explicit generation horizon. If None, estimate from instance.
    horizon_factor
        Used only if horizon is None.
    machine_models
        Reliability model dictionary with optional "default" and per-machine entries.
    allow_overlap
        If False, overlapping holes on the same machine are pushed forward.
    """
    validate_instance_dict(instance_dict)

    rng = random.Random(seed)
    num_machines = get_num_machines(instance_dict)

    if horizon is None:
        horizon = estimate_horizon(instance_dict, horizon_factor=horizon_factor)

    holes_by_machine: dict[int, list[tuple[int, int]]] = {}

    for m in range(num_machines):
        holes = generate_holes_for_machine(
            machine_id=m,
            horizon=horizon,
            rng=rng,
            machine_models=machine_models,
            without_fixed_dates=without_fixed_dates,
            allow_overlap=allow_overlap,
        )
        holes_by_machine[m] = holes

    return holes_by_machine

In [9]:
def instance_dict_to_jspmu_text(
    instance_dict: dict[str, Any],
    holes_by_machine: dict[int, list[tuple[int, int]]],
) -> str:
    """
    Format:
      num_jobs num_machines
      machine duration machine duration ...
      ...
      [MACHINE_HOLES]
      m 0 0
      or
      m k s1 d1 s2 d2 ... sk dk
    """
    validate_instance_dict(instance_dict)

    duration_matrix = instance_dict["duration_matrix"]
    machines_matrix = instance_dict["machines_matrix"]

    num_jobs = get_num_jobs(instance_dict)
    num_machines = get_num_machines(instance_dict)

    lines: list[str] = []

    # Header
    lines.append(f"{num_jobs} {num_machines}")

    # JSP operations
    for mac_row, dur_row in zip(machines_matrix, duration_matrix):
        tokens: list[str] = []
        for machine, duration in zip(mac_row, dur_row):
            tokens.append(str(machine))
            tokens.append(str(duration))
        lines.append(" ".join(tokens))

    # Machine holes
    lines.append("")
    lines.append("[MACHINE_HOLES]")

    for m in range(num_machines):
        holes = holes_by_machine.get(m, [])

        if len(holes) == 0:
            lines.append(f"{m} 0 0")
            continue

        tokens = [str(m), str(len(holes))]
        for start, duration in holes:
            tokens.append(str(start))
            tokens.append(str(duration))

        lines.append(" ".join(tokens))

    return "\n".join(lines) + "\n"


def write_jspmu_instance(
    instance_dict: dict[str, Any],
    output_path: str | Path,
    seed: int,
    without_fixed_dates: bool = False,
    horizon: int | None = None,
    horizon_factor: float = 1.0,
    machine_models: dict[Any, dict[str, Any]] | None = None,
    allow_overlap: bool = False,
) -> Path:
    """
    End-to-end generator.
    """
    holes_by_machine = generate_machine_holes(
        instance_dict=instance_dict,
        seed=seed,
        without_fixed_dates=without_fixed_dates,
        horizon=horizon,
        horizon_factor=horizon_factor,
        machine_models=machine_models,
        allow_overlap=allow_overlap,
    )

    text = instance_dict_to_jspmu_text(
        instance_dict=instance_dict,
        holes_by_machine=holes_by_machine,
    )

    output_path = Path(output_path)
    output_path.write_text(text, encoding="utf-8")
    return output_path

# test instance generation

In [10]:
benchmark_instance = load_benchmark_instance("abz5")

In [11]:
instance_dict = benchmark_instance.to_dict()

In [12]:
dist = "lognormal"
miu = 35
sigma = 0.3
machine_models = {
    "default": {
        "interarrival": {"dist": "exponential", "mtbf": 300},
        "duration": {"dist": dist, "mean": miu, "sigma": sigma},
    }
}

In [13]:
write_jspmu_instance(
    instance_dict=instance_dict,
    output_path="la01_mu_fixed.txt",
    seed=42,
    without_fixed_dates=False,
    horizon_factor=1.0,
    machine_models=machine_models,
)

PosixPath('la01_mu_fixed.txt')

# generate multiple instances from literature

In [14]:
import os
import zipfile
from tqdm import tqdm

In [15]:
instances_df_names = pd.read_csv('instances_basic_data.csv')

In [16]:
instances_df_names.head()

,id,ref,jobs,machines,lb,lb ref,bks,bks ref,t(bks) in s,t(bks) ref
0,abz5,ABZ,10,10,1234,AC,1234,AC,0.04,AZ
1,abz6,ABZ,10,10,943,AC,943,AC,0.03,AZ
2,abz7,ABZ,20,15,656,M,656,H,1000.00,H
3,abz8,ABZ,20,15,648,VLS,665,H,1000.00,H
4,abz9,ABZ,20,15,678,KNF,678,ZSR,3.25,AZ


In [17]:
count = 0
selected_instances = []
for idx,row in instances_df_names.iterrows():
    if row['jobs'] <= 10 and row['machines'] <= 10:
        print(row['id'])
        selected_instances.append(row['id'])
        count += 1
print(count)

abz5
abz6
ft06
ft10
la01
la02
la03
la04
la05
la16
la17
la18
la19
la20
orb01
orb02
orb03
orb04
orb05
orb06
orb07
orb08
orb09
orb10
24


In [18]:
!rm -rf generated_jspmu_instances_literature

In [19]:
output_dir = "generated_jspmu_instances_literature"
zip_name = "generated_jspmu_instances_litarature.zip"

In [20]:
seed = 42
without_fixed_dates = False
horizon_factor = 1.0

mtbf = 300
dist = "lognormal"
miu = 35
sigma = 0.3
machine_models = {
    "default": {
        "interarrival": {"dist": "exponential", "mtbf": mtbf},
        "duration": {"dist": dist, "mean": miu, "sigma": sigma},
    }
}

In [21]:
os.makedirs(output_dir, exist_ok=True)

In [22]:
generated_files = []

for instance_name in tqdm(selected_instances):
    try:
        benchmark_instance = load_benchmark_instance(instance_name)
        instance_dict = benchmark_instance.to_dict()

        output_path = os.path.join(output_dir, f"{instance_name}_mu_mtbf{mtbf}_durlognorm{miu}_{str(sigma).replace('.',"")}.jspmu")

        write_jspmu_instance(
            instance_dict=instance_dict,
            output_path=output_path,
            seed=seed,
            without_fixed_dates=without_fixed_dates,
            horizon_factor=horizon_factor,
            machine_models=machine_models,
        )

        generated_files.append(output_path)
        # print(f"Generated: {output_path}")

    except Exception as e:
        print(f"Failed for {instance_name}: {e}")

100%|██████████| 24/24 [00:00<00:00, 1060.48it/s]


In [23]:
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in generated_files:
        arcname = os.path.basename(file_path)
        zipf.write(file_path, arcname=arcname)

print(f"\nZIP created: {zip_name}")


ZIP created: generated_jspmu_instances_litarature.zip


# Re-scale instance generator

In [25]:
from copy import deepcopy
import random

In [36]:
def subsample_instance(
    instance_dict: dict,
    holes_by_machine: dict[int, list[tuple[int, int]]],
    max_jobs: int | None = None,
    max_ops_per_job: int | None = None,
    max_machines: int | None = None,
    seed: int | None = None,
) -> tuple[dict, dict[int, list[tuple[int, int]]]]:
    """
    Subsample a job-shop instance to reduce its size.

    Steps (applied in order):
      1. Sample at most `max_jobs` jobs (random subset).
      2. Truncate each job to at most `max_ops_per_job` operations
         (keep the first N in sequence order).
      3. Sample at most `max_machines` machines (random subset),
         then reassign operations on dropped machines to a random
         kept machine. Holes for dropped machines are discarded;
         holes for kept machines are remapped to new IDs.

    Parameters
    ----------
    instance_dict : dict
        Must contain at least:
          - "machines_matrix":  list[list[int]]  (jobs × ops → machine id)
          - "duration_matrix":  list[list[int]]  (jobs × ops → duration)
    holes_by_machine : dict[int, list[tuple[int, int]]]
        {old_machine_id: [(start, duration), ...], ...}
    max_jobs : int | None
        Maximum number of jobs to keep.  None = keep all.
    max_ops_per_job : int | None
        Maximum operations per job.  None = keep all.
    max_machines : int | None
        Maximum number of machines to keep.  None = keep all.
    seed : int | None
        Random seed for reproducibility.

    Returns
    -------
    new_instance_dict, new_holes_by_machine
        The subsampled instance and corresponding holes (with
        machine IDs remapped to 0 .. len(kept_machines)-1).
    """
    rng = random.Random(seed)
    inst = deepcopy(instance_dict)

    machine_matrix = inst["machines_matrix"]
    duration_matrix = inst["duration_matrix"]
    n_jobs = len(machine_matrix)
    n_machines = len({m for row in machine_matrix for m in row})

    # ------------------------------------------------------------------
    # 1) Subsample jobs
    # ------------------------------------------------------------------
    job_indices = list(range(n_jobs))

    if max_jobs is not None and max_jobs < n_jobs:
        job_indices = sorted(rng.sample(job_indices, max_jobs))

    machine_matrix = [machine_matrix[j] for j in job_indices]
    duration_matrix = [duration_matrix[j] for j in job_indices]

    # ------------------------------------------------------------------
    # 2) Truncate operations per job (keep first N)
    # ------------------------------------------------------------------
    if max_ops_per_job is not None:
        machine_matrix = [row[:max_ops_per_job] for row in machine_matrix]
        duration_matrix = [row[:max_ops_per_job] for row in duration_matrix]

    # ------------------------------------------------------------------
    # 3) Subsample machines
    # ------------------------------------------------------------------
    # Discover all machine IDs currently referenced
    all_machines = sorted({m for row in machine_matrix for m in row})

    if max_machines is not None and max_machines < len(all_machines):
        kept_machines = sorted(rng.sample(all_machines, max_machines))
    else:
        kept_machines = all_machines

    # Build remapping:  old_id -> new_id  (0-based, contiguous)
    old_to_new = {old: new for new, old in enumerate(kept_machines)}

    # Reassign operations on dropped machines to a random kept machine
    new_machine_matrix = []
    for row in machine_matrix:
        new_row = []
        for m in row:
            if m in old_to_new:
                new_row.append(old_to_new[m])
            else:
                # Dropped machine → reassign randomly
                new_row.append(old_to_new[rng.choice(kept_machines)])
        new_machine_matrix.append(new_row)

    # ------------------------------------------------------------------
    # 4) Remap holes to kept machines only
    # ------------------------------------------------------------------
    new_holes: dict[int, list[tuple[int, int]]] = {}

    for old_m in kept_machines:
        new_m = old_to_new[old_m]
        if old_m in holes_by_machine:
            new_holes[new_m] = deepcopy(holes_by_machine[old_m])
        # If there were no holes for this machine, we just skip it
        # (or you could insert an empty list — depends on your downstream code)

    # ------------------------------------------------------------------
    # 5) Build the new instance dict
    # ------------------------------------------------------------------
    inst["machines_matrix"] = new_machine_matrix
    inst["duration_matrix"] = duration_matrix

    return inst, new_holes

In [27]:
def _flatten(matrix: list[list[int]]) -> list[int]:
    return [x for row in matrix for x in row]


def _min_max_rescale(
    values: list[float],
    new_min: float,
    new_max: float,
    integer: bool = True,
    floor_at: int | None = 1,
) -> list[int] | list[float]:
    """
    Min-max rescale a list of values into [new_min, new_max].

    If all values are equal, map everything to the midpoint.
    """
    if not values:
        return []

    old_min = min(values)
    old_max = max(values)

    if old_max == old_min:
        midpoint = 0.5 * (new_min + new_max)
        out = [midpoint for _ in values]
    else:
        out = [
            new_min + (x - old_min) * (new_max - new_min) / (old_max - old_min)
            for x in values
        ]

    if integer:
        out = [int(round(x)) for x in out]
        if floor_at is not None:
            out = [max(floor_at, x) for x in out]

    return out


def rescale_instance_and_holes(
    instance_dict: dict[str, Any],
    holes_by_machine: dict[int, list[tuple[int, int]]],
    op_min: int,
    op_max: int,
    hole_dur_min: int,
    hole_dur_max: int,
    keep_unfixed_holes_as_minus_one: bool = True,
    sort_holes_after_rescaling: bool = True,
) -> tuple[dict[str, Any], dict[int, list[tuple[int, int]]]]:
    """
    Rescale:
      1. operation durations into [op_min, op_max]
      2. machine-hole start dates into [0, sum(new operation durations)]
      3. machine-hole durations into [hole_dur_min, hole_dur_max]

    Parameters
    ----------
    instance_dict
        Original job-shop instance dictionary.
    holes_by_machine
        Dict like {machine: [(start, duration), ...], ...}
        If a hole has start = -1, it is treated as an unfixed-date hole.
    op_min, op_max
        New min/max for operation durations.
    hole_dur_min, hole_dur_max
        New min/max for hole durations.
    keep_unfixed_holes_as_minus_one
        If True, holes with start=-1 stay at -1 and only their durations are rescaled.
    sort_holes_after_rescaling
        If True, fixed-date holes are sorted by start within each machine.

    Returns
    -------
    new_instance_dict, new_holes_by_machine
    """
    if op_min <= 0 or op_max <= 0 or op_min > op_max:
        raise ValueError("Require 0 < op_min <= op_max.")

    if hole_dur_min <= 0 or hole_dur_max <= 0 or hole_dur_min > hole_dur_max:
        raise ValueError("Require 0 < hole_dur_min <= hole_dur_max.")

    new_instance = deepcopy(instance_dict)
    new_holes = deepcopy(holes_by_machine)

    # --------------------------------------------------
    # 1) Rescale operation durations
    # --------------------------------------------------
    old_durations = _flatten(instance_dict["duration_matrix"])
    new_durations = _min_max_rescale(
        old_durations,
        new_min=op_min,
        new_max=op_max,
        integer=True,
        floor_at=1,
    )

    new_duration_matrix = []
    idx = 0
    for row in instance_dict["duration_matrix"]:
        new_row = new_durations[idx : idx + len(row)]
        new_duration_matrix.append(new_row)
        idx += len(row)

    new_instance["duration_matrix"] = new_duration_matrix

    # New max date for holes = total new operation workload
    new_total_workload = sum(sum(row) for row in new_duration_matrix)

    # --------------------------------------------------
    # 2) Collect fixed-date starts and all hole durations
    # --------------------------------------------------
    fixed_starts = []
    duration_values = []

    for machine, holes in holes_by_machine.items():
        for start, dur in holes:
            duration_values.append(dur)
            if start != -1:
                fixed_starts.append(start)

    # Rescale fixed starts into [0, new_total_workload]
    rescaled_fixed_starts = _min_max_rescale(
        fixed_starts,
        new_min=0,
        new_max=new_total_workload,
        integer=True,
        floor_at=0,
    )

    # Rescale hole durations into [hole_dur_min, hole_dur_max]
    rescaled_hole_durations = _min_max_rescale(
        duration_values,
        new_min=hole_dur_min,
        new_max=hole_dur_max,
        integer=True,
        floor_at=1,
    )

    # --------------------------------------------------
    # 3) Rebuild holes
    # --------------------------------------------------
    start_idx = 0
    dur_idx = 0

    rebuilt_holes: dict[int, list[tuple[int, int]]] = {}

    for machine, holes in new_holes.items():
        rebuilt = []

        for start, _dur in holes:
            new_dur = rescaled_hole_durations[dur_idx]
            dur_idx += 1

            if start == -1 and keep_unfixed_holes_as_minus_one:
                new_start = -1
            elif start == -1:
                # Keep as -1 anyway unless you later want a rule to assign a date
                new_start = -1
            else:
                new_start = rescaled_fixed_starts[start_idx]
                start_idx += 1

                # Optional safety clamp
                new_start = min(new_start, new_total_workload)

            rebuilt.append((new_start, new_dur))

        if sort_holes_after_rescaling:
            fixed = [h for h in rebuilt if h[0] != -1]
            unfixed = [h for h in rebuilt if h[0] == -1]
            fixed.sort(key=lambda x: x[0])
            rebuilt = fixed + unfixed

        rebuilt_holes[machine] = rebuilt

    return new_instance, rebuilt_holes

In [28]:
def clean_rescaled_holes(
    holes_by_machine: dict[int, list[tuple[int, int]]],
    max_start: int,
    min_gap: int = 0,
) -> dict[int, list[tuple[int, int]]]:
    cleaned = {}

    for machine, holes in holes_by_machine.items():
        fixed = [h for h in holes if h[0] != -1]
        unfixed = [h for h in holes if h[0] == -1]

        fixed.sort(key=lambda x: x[0])

        repaired = []
        current_end = 0

        for start, dur in fixed:
            start = max(start, current_end + min_gap)
            if start > max_start:
                continue

            dur = max(1, dur)
            repaired.append((start, dur))
            current_end = start + dur

        cleaned[machine] = repaired + unfixed

    return cleaned

In [29]:
def keep_first_k_holes_per_machine(
    holes_by_machine: dict[int, list[tuple[int, int]]],
    k: int,
    sort_fixed_holes: bool = True,
) -> dict[int, list[tuple[int, int]]]:
    """
    Keep only the first k holes per machine.

    Rules:
    - if a machine has <= k holes, keep them all
    - if a machine has > k holes, keep only the first k
    - fixed-date holes are optionally sorted by start time
    - unfixed holes with start = -1 are placed after fixed holes
    """
    if k < 0:
        raise ValueError("k must be >= 0.")

    trimmed = {}

    for machine, holes in holes_by_machine.items():
        if sort_fixed_holes:
            fixed = [h for h in holes if h[0] != -1]
            unfixed = [h for h in holes if h[0] == -1]

            fixed.sort(key=lambda x: x[0])
            ordered = fixed + unfixed
        else:
            ordered = list(holes)

        trimmed[machine] = ordered[:k]

    return trimmed

In [30]:
!rm -rf generated_jspmu_instances_literature_rescaled

In [31]:
output_dir = "generated_jspmu_instances_literature_rescaled"
zip_name = "generated_jspmu_instances_literature_rescaled_3.zip"

In [32]:
seed = 42
without_fixed_dates = False
horizon_factor = 1.0

mtbf = 300
dist = "lognormal"
miu = 35
sigma = 0.3
machine_models = {
    "default": {
        "interarrival": {"dist": "exponential", "mtbf": mtbf},
        "duration": {"dist": dist, "mean": miu, "sigma": sigma},
    }
}


In [33]:
os.makedirs(output_dir, exist_ok=True)

In [37]:
op_min = 1
op_max = 3

hole_dur_min = 1
hole_dur_max = 1

max_jobs = 3
max_ops_per_job = 3
max_machines = 3

k_unavailabilities_per_machine = 2

generated_files = []

for instance_name in tqdm(selected_instances):
    try:
        benchmark_instance = load_benchmark_instance(instance_name)
        instance_dict = benchmark_instance.to_dict()


        # --------------------------------------------------
        # 1) Generate machine holes
        # --------------------------------------------------
        holes_by_machine = generate_machine_holes(
            instance_dict=instance_dict,
            seed=seed,
            without_fixed_dates=without_fixed_dates,
            horizon_factor=horizon_factor,
            machine_models=machine_models,
        )

        # --------------------------------------------------
        # 2) Subsample + Rescale instance + holes
        # --------------------------------------------------

        # Subsample first
        instance_dict, holes_by_machine = subsample_instance(
            instance_dict=instance_dict,
            holes_by_machine=holes_by_machine,
            max_jobs=max_jobs,
            max_ops_per_job=max_ops_per_job,
            max_machines=max_machines,
            seed=seed,
        )

        scaled_instance_dict, scaled_holes_by_machine = rescale_instance_and_holes(
            instance_dict=instance_dict,
            holes_by_machine=holes_by_machine,
            op_min=op_min,                  # e.g. 1
            op_max=op_max,                  # e.g. 5
            hole_dur_min=hole_dur_min,      # e.g. 2
            hole_dur_max=hole_dur_max,      # e.g. 6
            keep_unfixed_holes_as_minus_one=True,
            sort_holes_after_rescaling=True,
        )

        # --------------------------------------------------
        # 3) Optional cleaning step after rescaling
        # --------------------------------------------------
        max_start = sum(sum(row) for row in scaled_instance_dict["duration_matrix"])

        scaled_holes_by_machine = clean_rescaled_holes(
            holes_by_machine=scaled_holes_by_machine,
            max_start=max_start,
            min_gap=0,
        )

        scaled_holes_by_machine = keep_first_k_holes_per_machine(
            holes_by_machine=scaled_holes_by_machine,
            k=k_unavailabilities_per_machine,
        )

        # --------------------------------------------------
        # 4) Convert to text and save
        # --------------------------------------------------
        output_filename = (
            f"{instance_name}"
            f"_mu_mtbf{mtbf}"
            f"_durlognorm{miu}_{str(sigma).replace('.', '')}"
            f"_op{op_min}-{op_max}"
            f"_hole{hole_dur_min}-{hole_dur_max}"
            f".jspmu"
        )

        output_path = os.path.join(output_dir, output_filename)

        jspmu_text = instance_dict_to_jspmu_text(
            instance_dict=scaled_instance_dict,
            holes_by_machine=scaled_holes_by_machine,
        )

        with open(output_path, "w", encoding="utf-8") as f:
            f.write(jspmu_text)

        generated_files.append(output_path)
        # print(f"Generated: {output_path}")

    except Exception as e:
        print(f"Failed for {instance_name}: {e}")

100%|██████████| 24/24 [00:00<00:00, 848.63it/s]


In [38]:
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in generated_files:
        arcname = os.path.basename(file_path)
        zipf.write(file_path, arcname=arcname)

print(f"\nZIP created: {zip_name}")


ZIP created: generated_jspmu_instances_literature_rescaled_3.zip
